# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [2]:
%uv pip install scanpy==1.11.5 anndata==0.11.4 pandas==2.3.3 numpy==2.2.6 scikit-learn==1.7.2 umap-learn==0.5.12 leidenalg==0.11.0 igraph==1.0.0

Using Python 3.12.6 environment at: /usr/local
Resolved 38 packages in 222ms
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------     0 B/9.94 KiB
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
texttable  ------------------------------ 10.52 KiB/10.52 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
texttable  ------------------------------ 10.52 KiB/10.52 KiB
natsort    ------------------------------ 16.00 KiB/37.37 KiB
⠙ Preparing packages... (0/14)
legacy-api-wrap ------------------------------ 9.94 KiB/9.94 KiB
texttable  ------------------------------ 10

In [3]:
import pandas as pd
import numpy as np
import scanpy as sc
import os,gc, time
from tqdm import tqdm

sc.settings.verbosity = 2

In [4]:
data_path ="/mnt/scrna-chop-data/"

In [45]:
#Download ATAC files

!cd /mnt/scrna-chop-data/ && mkdir -p ataac_184547 && cd ataac_184547 && for f in GSE184547_atac.TF-RNA.correlations.tsv.gz GSE184547_distalATAC.RNA.correlations.tsv.gz GSE184547_atac.DAR.tsv.gz GSE184547_atac.peakAnno.tsv.gz GSE184547_rna.DEX.tsv.gz GSE184547_rna.geneAnno.tsv.gz GSE184547_atac.meta.tsv.gz GSE184547_promoterATAC.RNA.correlations.tsv.gz GSE184547_atac.chromVAR.Zscore.tsv.gz; do echo "Downloading $f..."; wget -q "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE184nnn/GSE184547/suppl/$f" -O "$f"; done && echo "=== Done ===" && ls -lh *.gz

=== Done ===
-rw-r--r-- 1 root root 7.9M Sep 10  2021 GSE184547_atac.DAR.tsv.gz
-rw-r--r-- 1 root root  11K Sep 10  2021 GSE184547_atac.TF-RNA.correlations.tsv.gz
-rw-r--r-- 1 root root 167K Sep 10  2021 GSE184547_atac.chromVAR.Zscore.tsv.gz
-rw-r--r-- 1 root root  586 Sep 10  2021 GSE184547_atac.meta.tsv.gz
-rw-r--r-- 1 root root 5.4M Sep 10  2021 GSE184547_atac.peakAnno.tsv.gz
-rw-r--r-- 1 root root  15M Sep 10  2021 GSE184547_distalATAC.RNA.correlations.tsv.gz
-rw-r--r-- 1 root root 1.3M Sep 10  2021 GSE184547_promoterATAC.RNA.correlations.tsv.gz
-rw-r--r-- 1 root root 507K Jul 25  2023 GSE184547_rna.DEX.tsv.gz
-rw-r--r-- 1 root root 457K Sep 10  2021 GSE184547_rna.geneAnno.tsv.gz


In [1]:
# Inspect the DAR file
print("=== ATAC DAR file ===", flush=True)
dar = pd.read_csv("/mnt/scrna-chop-data/ataac_184547/GSE184547_atac.DAR.tsv.gz", sep='\t', nrows=10)
print(f"Columns: {list(dar.columns)}")
print(f"Shape (first 10 rows): {dar.shape}")
print(dar.head(5).to_string())
print()

# Inspect peak annotation
print("=== ATAC peakAnno file ===", flush=True)
anno = pd.read_csv("/mnt/scrna-chop-data/ataac_184547/GSE184547_atac.peakAnno.tsv.gz", sep='\t', nrows=10)
print(f"Columns: {list(anno.columns)}")
print(f"Shape (first 10 rows): {anno.shape}")
print(anno.head(5).to_string())
print()

# Inspect RNA DEX
print("=== RNA DEX file ===", flush=True)
rna_dex = pd.read_csv("/mnt/scrna-chop-data/ataac_184547/GSE184547_rna.DEX.tsv.gz", sep='\t', nrows=10)
print(f"Columns: {list(rna_dex.columns)}")
print(f"Shape (first 10 rows): {rna_dex.shape}")
print(rna_dex.head(5).to_string())
print()

# Inspect meta
print("=== ATAC meta file ===", flush=True)
meta = pd.read_csv("/mnt/scrna-chop-data/ataac_184547/GSE184547_atac.meta.tsv.gz", sep='\t')
print(f"Columns: {list(meta.columns)}")
print(meta.to_string())

=== ATAC DAR file ===


NameError: name 'pd' is not defined

In [5]:
zscore = pd.read_csv(
    data_path + "ataac_184547/GSE184547_atac.chromVAR.Zscore.tsv.gz",
    sep="\t",
    index_col=0
)

meta = pd.read_csv(
    data_path + "ataac_184547/GSE184547_atac.meta.tsv.gz",
    sep="\t"
)

distal_corr = pd.read_csv(
    data_path + "ataac_184547/GSE184547_distalATAC.RNA.correlations.tsv.gz",
    sep="\t",
    index_col=0
)

tfrna = pd.read_csv(
    data_path + "ataac_184547/GSE184547_atac.TF-RNA.correlations.tsv.gz",
    sep="\t",
    index_col=0
)

# print(zscore.shape)
# print(zscore.head())
# print(zscore.columns)
# print(meta)

print(tfrna.shape)
print(tfrna.head())

(338, 3)
      corr.zscore.rna  corr.p.value  max.diff.zscore
TF                                                  
Sox4        -0.862197      0.000036        33.884823
Hdx         -0.009217      0.973992         6.867482
Irx2         0.771538      0.000755        27.035136
Irx4         0.544231      0.035957        26.461698
Irx5         0.804720      0.000297        23.639701


In [17]:
tfrna_sig = (
    tfrna.loc[
        (tfrna["corr.p.value"] < 0.05) &
        (tfrna["corr.zscore.rna"] > 0.5),
        ["corr.zscore.rna", "corr.p.value", "max.diff.zscore"]
    ]
    .sort_values("max.diff.zscore", ascending=False)
)

print(tfrna_sig)

        corr.zscore.rna  corr.p.value  max.diff.zscore
TF                                                    
Fosl2          0.896669      0.000006       114.253827
Vsx2           0.803099      0.000312        44.835077
Jund           0.792955      0.000421        42.013000
Pou6f1         0.882622      0.000013        41.024720
Jun            0.795323      0.000393        38.486987
...                 ...           ...              ...
Mecp2          0.622995      0.013103         7.865104
Cxxc1          0.572856      0.025608         7.637190
Atf3           0.683694      0.004948         7.561998
Hinfp          0.606070      0.016624         6.098407
Mybl1          0.829402      0.000131         5.532759

[79 rows x 3 columns]


In [18]:
tf_list = tfrna_sig.index.tolist()
#pd.DataFrame(tf_list).to_csv(data_path + "tfs_rna_corr_list.csv", index=False)

In [10]:
df = pd.read_csv("/mnt/scrna-chop-data/results/deg_temporal_with_specificity.csv")

In [14]:
df['ATAC_tf_rna_corr'] = df['gene'].isin(tf_list)

In [15]:
print(f"\nCandidates with ATAC promoter opening: {df['ATAC_tf_rna_corr'].sum()} / {len(df)}")
for profile in ['early_sustained', 'early_transient', 'late_onset']:
    sub = df[df['temporal_profile'] == profile]
    n_atac = sub['ATAC_tf_rna_corr'].sum()
    print(f"  {profile}: {n_atac}/{len(sub)} with ATAC tf_rna positive correlation")


Candidates with ATAC promoter opening: 11 / 1066
  early_sustained: 10/319 with ATAC tf_rna positive correlation
  early_transient: 0/256 with ATAC tf_rna positive correlation
  late_onset: 1/491 with ATAC tf_rna positive correlation


In [19]:
candidates = df[
    (df['temporal_profile'] == 'early_sustained') &
    (df['ATAC_tf_rna_corr']) & 
     (df['RGC_specific_non_glial'])
]

In [20]:
candidates

,gene,log2FC_12h,pval_12h,padj_12h,pct_injured_12h,pct_ctrl_12h,log2FC_1d,pval_1d,padj_1d,pct_injured_1d,...,max_log2FC,max_pct_injured,composite_score,rank_within_profile,glial_upregulated,glial_astrocyte_up,glial_microglia_up,glial_muller_up,RGC_specific_non_glial,ATAC_tf_rna_corr
4,Ddit3,0.080370,3.602879e-08,2.559940e-07,0.431929,0.411304,1.979698,0.000000e+00,0.000000e+00,0.744924,...,4.424330,0.967856,0.669823,5.0,False,False,False,False,True,True
5,Sox11,1.128386,1.896344e-93,7.850865e-92,0.319837,0.212165,2.140324,0.000000e+00,0.000000e+00,0.517604,...,3.942556,0.824033,0.662847,6.0,False,False,False,False,True,True
16,Jun,1.995054,0.000000e+00,0.000000e+00,0.817255,0.436686,2.292038,0.000000e+00,0.000000e+00,0.881999,...,3.156958,0.960593,0.638154,17.0,False,False,False,False,True,True
26,Crem,0.164993,1.863358e-01,5.745507e-01,0.167731,0.164477,1.986385,0.000000e+00,0.000000e+00,0.462364,...,3.019637,0.620092,0.623372,27.0,False,False,False,False,True,True
177,Cebpd,2.424048,4.327840e-16,4.517799e-15,0.066508,0.014536,2.745420,2.601553e-38,3.567243e-37,0.097696,...,2.745420,0.097696,0.550997,178.0,False,False,False,False,True,True
314,Tead1,0.918937,1.969655e-01,6.021442e-01,0.023777,0.015598,1.233763,1.088659e-04,4.561789e-04,0.040541,...,1.389936,0.052567,0.525860,315.0,False,False,False,False,True,True
